# Reranker Experiment

This experiment tests a two-stage retrieval pipeline: PubMedBERT embeddings first generate a candidate set, then a biomedical cross-encoder reranks those candidates.

The candidate generator is `neuml/biomedbert-base-embeddings`. The reranker is `neuml/biomedbert-base-reranker`, a cross-encoder based on BiomedBERT and trained for semantic ranking over biomedical literature. The model card explicitly describes its use for reranking and semantic search. citeturn0search0turn0search9

The key engineering trade-off is deliberate: dense embeddings are efficient for broad candidate retrieval, while a cross-encoder can score query-document pairs more precisely but is more expensive. Therefore the reranker is applied only to the top candidate set rather than to the entire corpus.

## Hypothesis

The embedding comparison showed that PubMedBERT can retrieve difficult biomedical sources but can also produce ranking regressions. A reranker should improve the ordering of the retrieved candidates by evaluating the query and passage jointly.

The experiment tests candidate sizes of 5, 10, 20, and 50 so that we can measure the trade-off between reranking depth and retrieval quality.

## Experimental design

1. Encode the 979 chunks with PubMedBERT embeddings.
2. Encode the 500 benchmark questions with the same embedding model.
3. Retrieve the top K candidates by cosine similarity.
4. Score only those candidates with `neuml/biomedbert-base-reranker`.
5. Reorder candidates by cross-encoder score.
6. Measure Recall@1, Recall@3, Recall@5, MRR, and top-5 misses.

The source-level relevance definition remains unchanged: a retrieved chunk is relevant when its PubMed ID matches the query's source PubMed ID.

## Run

From the repository root:

```bash
python src/reranker_experiment.py
```

The experiment writes results to `data/reranker_results.json`.

## Interpretation

The main objective is to improve ranking without sacrificing candidate recall.

If increasing candidate K improves Recall@1/MRR while preserving high Recall@5, choose the smallest K that gives a useful improvement. This keeps the cross-encoder's computational cost under control.

A reranker cannot recover a document that the embedding stage fails to place inside the candidate set. Therefore the candidate K experiment also tells us whether the remaining retrieval misses are caused by candidate generation or by poor ranking within the candidate set.

## Results

The reranker improved the baseline retrieval results, with the strongest overall result at a candidate pool of 50:

| Candidates | Recall@1 | Recall@3 | Recall@5 | MRR | Top-5 misses |
|---:|---:|---:|---:|---:|---:|
| 5 | 0.978 | 0.990 | 0.992 | 0.984 | 4 |
| 10 | 0.978 | 0.990 | 0.992 | 0.984 | 4 |
| 20 | 0.976 | 0.992 | 0.994 | 0.984 | 3 |
| 50 | 0.976 | 0.994 | 0.996 | 0.985 | 2 |

For comparison, the PubMedBERT embedding-only benchmark was Recall@1 0.972, Recall@3 0.984, Recall@5 0.992, and MRR 0.979, with 4 top-five misses.

The reranker therefore improves the retrieval system overall. Increasing the candidate pool from 5/10 to 20/50 improves Recall@3 and Recall@5 and reduces the number of top-five misses. The improvement from 20 to 50 is smaller, but 50 gives the best measured MRR and Recall@3/5.


## Engineering decision

Use a two-stage retrieval architecture: PubMedBERT dense retrieval for candidate generation followed by the biomedical cross-encoder reranker.

For the current project, use a candidate pool of 20 as the initial operating point. It captures almost all of the measured benefit of 50 candidates while requiring substantially fewer cross-encoder query-document evaluations. The 50-candidate configuration remains a useful benchmark and could be revisited if later end-to-end QA evaluation shows that the additional recall is valuable.

This is a deliberate engineering trade-off rather than choosing the numerically highest metric without considering cost. The 20-candidate configuration has Recall@5 0.994 and 3 top-five misses, compared with 0.996 and 2 misses for 50 candidates, while using less than half as many reranker evaluations per query.


## Retrieval phase conclusion

The retrieval work has now established a concrete pipeline rather than a single embedding lookup:

`query → PubMedBERT candidate retrieval → biomedical cross-encoder reranking → ranked context`

The experiments also provide evidence for the main design choices: context-preserving chunking, biomedical-domain embeddings, and a reranking stage for resolving semantically similar candidates.

The next major project stage should therefore move beyond retrieval into the generation layer and connect the ranked retrieved context to an LLM for answer generation. Retrieval should not be further optimized unless end-to-end evaluation exposes a specific retrieval problem.